# 00 — Exploratory Data Analysis (EDA descriptive)

**Notebook id**: `00_eda` → vault `03.1-EDA-descriptive.md`

Cohort flow, a **data-anomaly audit** of the source dates, **Table 1 (SMD)**, baseline (S1) equivalence, raw per-compartment frequency tables on the **native 0–3 scale**, the **binary** status of PTI/CFI, the cohort of **n = 69** analysable for PF progression, and a note on the **{0, 1, ≥2} collapse** used for modelling. No inferential decision here — description only.

In [ ]:
# --- Setup (idempotent, fresh-kernel reproducible) ---
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd

from constants import (
    RANDOM_SEED, SITES, SITES_PF, SITES_FT, SITES_BINARY, SITES_ORDINAL,
    GROUPS, BLOCKS, N_TOTAL, N_TOTAL_ANALYSABLE, N_MENISCUS, N_CYCLOPS,
    SCORE_MAX, SCORE_MAX_COLLAPSED,
)
import loaders
import preprocessing as pp
import tests_freq as tf
import reporting as rpt
import bayes_models as bm
import viz

np.random.seed(RANDOM_SEED)
viz.set_pub_style()


In [ ]:
# --- Load & preprocess (canonical pipeline) ---
df = loaders.load_combined()
df = pp.apply_date_hygiene(df)      # composite-key (group, anonyme) date hygiene
df = pp.add_derived(df)            # lesion_pf/ft, female, deltas, worsened_pf, ...
wide = pp.to_wide(df)             # one row per patient (group, anonyme)
patient = pp.to_patient(df)       # static covariates per patient

# Patient-level covariates joined onto the wide outcomes (for H3 / sensitivity).
_cov = [c for c in ['group','anonyme','female','sexe','pivot_pivot_contact',
                    'travail_physique','tabac','age_at_trauma','imc','taille','poids']
        if c in patient.columns]
merged = wide.merge(patient[_cov], on=['group','anonyme'], how='left')

print('long:', df.shape, '| wide:', wide.shape, '| patient:', patient.shape,
      '| merged:', merged.shape)
# Composite-key sentinel: 19 Anonyme ids are reused across the two sheets.
assert (df.groupby(['group','anonyme']).size() == 2).all(), 'composite key broken'


## 0. Data-anomaly audit (source dates only — NOT the cartilage Δ)

`pp.detect_date_anomalies` is run on the **RAW combined frame** (before `apply_date_hygiene`) so the divergences are still visible. It flags two kinds of source-date issue that pollute *age / delay / baseline / H3 / H4* but **never** the patellofemoral Δ (the score columns carry no dates):

- **static-date drift** — `date_de_naissance` / `date_du_trauma` differing between a patient's S1 and S2 rows (e.g. patient #9, a trauma-date drift of ~61 d; hygiene resolves it by `min`, the flag is for manual fixing);
- **negative trauma→surgery** — surgery dated *before* the trauma (impossible; #25 / #38), a placeholder/typo that `add_derived` now coerces to NaN.

These rows form the exclusion list for the delay sensitivity in 06; they leave `inter_surgery_d` (an S2−S1 *difference* of two `date_chir`) intact.

In [ ]:
anomalies = pp.detect_date_anomalies(loaders.load_combined())
print(f"date anomalies found: {len(anomalies)}")
if len(anomalies):
    print(anomalies.to_string(index=False))
else:
    print('(none)')
print()
print('Breakdown by kind:')
print(anomalies['kind'].value_counts() if len(anomalies) else '(none)')
print()
print('NB: none of these touch the cartilage scores / the PF delta, and '
      'inter_surgery_d (a date_chir S2-S1 difference) is unaffected.')


## 1. Cohort flow & missingness

138 long rows = 69 patients × 2 surgeries. After patient 25 was reclassified cyclops→meniscus (2026-05-29) and its operated-today S2 data completed, every patient carries a usable PF outcome → **n = 69** (49 cyclops + 20 meniscus) analysable for progression.

In [ ]:
print(f"Patients      : meniscus={N_MENISCUS}, cyclops={N_CYCLOPS}, total={N_TOTAL}")
print(f"Long rows     : {len(df)} (expected 138)")
print(f"Wide rows     : {len(wide)} (expected 69)")
print()
print('Missingness on lesion sites (long):')
print(df[SITES].isna().sum())
print()
# Exclusion flow for the PF progression outcome.
pf_ok = wide['delta_lesion_pf'].notna()
print('Analysable for PF progression (delta_lesion_pf not NaN):')
print(wide.loc[pf_ok, 'group'].value_counts())
print(f"=> n analysable = {int(pf_ok.sum())} (expected {N_TOTAL_ANALYSABLE})")
assert int(pf_ok.sum()) == N_TOTAL_ANALYSABLE


## 2. Table 1 — baseline cohort summary (SMD, Austin)

Continuous covariates: median [IQR] + Mann–Whitney p + standardised mean difference. Categorical: n (%) + Fisher + binary SMD. The SMD is the balance metric (Austin 2009); large |SMD| flags an imbalance to carry into a sensitivity analysis (e.g. age), not the primary model.

In [ ]:
table1 = rpt.make_table1(patient)
print(table1.to_string(index=False))


## 3. Raw per-compartment frequency tables (native 0–3 scale)

Native grades are kept here for description. Note PTI and CFI carry **zero** events at grade ≥ 2 → they are strictly **binary** and are modelled with a Bernoulli likelihood in M3; the other four sites keep an ordinal cumulative logit on the collapsed {0, 1, ≥2} scale.

In [ ]:
for grp, sub in df.groupby('group'):
    print(f"=== {grp} (native 0-3) ===")
    counts = pd.concat({s: sub[s].value_counts().sort_index() for s in SITES},
                       axis=1).fillna(0).astype(int)
    print(counts)
    print()

# Binary vs ordinal classification used by M3.
print('Modelling families:')
print('  Bernoulli (binary, 0 events >=2):', SITES_BINARY)
print('  Cumulative logit (ordinal {0,1,>=2}):', SITES_ORDINAL)
print()
# Confirm the binary sites really never reach grade >= 2 in the data.
for s in SITES_BINARY:
    mx = int(pd.to_numeric(df[s], errors='coerce').max())
    print(f'  max({s}) = {mx}  -> binary OK' if mx <= 1 else f'  max({s}) = {mx}  !!')


## 4. The {0, 1, ≥2} collapse (information-neutral on the PF signal)

Modelling uses the collapsed scale (grade 3 → 2). Below we confirm the collapse barely moves the patellofemoral effect size — it removes a near-empty top category, it does not remove signal.

In [ ]:
# Native vs collapsed PF block delta, Cliff's delta cyclops vs meniscus.
df_coll = pp.collapse_scores(df, SITES)
wide_coll = pp.to_wide(pp.add_derived(df_coll))

def _cliff_pf(w):
    c = w.loc[w.group=='cyclops','delta_lesion_pf'].dropna().astype(float).values
    m = w.loc[w.group=='meniscus','delta_lesion_pf'].dropna().astype(float).values
    return tf.cliffs_delta(c, m)

print(f"Cliff delta PF  native(0-3)   = {_cliff_pf(wide):+.3f}")
print(f"Cliff delta PF  collapsed     = {_cliff_pf(wide_coll):+.3f}")
print("(near-identical => collapse is information-neutral on the PF signal)")


## 5. Distribution of `lesion_pf` / `lesion_ft` by (group × time)

In [ ]:
for col in ['lesion_pf', 'lesion_ft', 'lesion_total']:
    if col in df.columns:
        print(f"--- {col} ---")
        print(df.groupby(['group','time'])[col].describe()[['count','mean','50%','min','max']])
        print()


## Sanity asserts

In [ ]:
assert patient.shape[0] == N_TOTAL == 69, 'Patient row count mismatch'
assert df['surgery_num'].value_counts().to_dict() == {1: 69, 2: 69}, 'S1/S2 imbalance'
print('All EDA asserts passed.')
